In [0]:
df = spark.read.table('bronze.loans')

In [0]:
df

In [0]:
from pyspark.sql.functions import col, sum

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

In [0]:
df = df.dropna()

In [0]:
## Check duplicate rows (Entire Row Duplicates)

df.groupby(df.columns).count().filter(col("count") > 1).show()

In [0]:
from pyspark.sql.functions import count 

dup_count = df.groupBy(df.columns).count().filter('count > 1').count() 


In [0]:
print(dup_count)

In [0]:
## Loan_id should be unique 

df.groupby("loan_id").count().filter('count > 1').show()

In [0]:
df = df.dropDuplicates()

In [0]:
%sql
use schema bronze

In [0]:
df.write.mode("overwrite").saveAsTable("pre_silver")

In [0]:
%sql
select * 
from bronze.pre_silver

In [0]:
%sql
-- create silver table

create or replace table silver.loans 
as

with silver_cte as
(select 
loan_id,
customer_id,
institution_id,
loan_amount,
loan_status,
interest_rate,
loan_tenure_months,
application_date,
disbursement_date,
maturity_date,
emi_amount,
round(loan_amount * (interest_rate/1200) * power((1 + interest_rate/1200) , loan_tenure_months) / 
(power((1 + interest_rate/1200), loan_tenure_months) - 1)) as emi_amount_calc,
purpose_of_loan

from bronze.pre_silver)

select 
loan_id,
customer_id,
institution_id,
loan_amount,
loan_status,
interest_rate,
loan_tenure_months,
application_date,
disbursement_date,
maturity_date,
round(emi_amount) as emi_amount,
round(loan_amount * (interest_rate/1200) * power((1 + interest_rate/1200) , loan_tenure_months) / 
(power((1 + interest_rate/1200), loan_tenure_months) - 1)) as emi_amount_calc,
purpose_of_loan

from silver_cte
where loan_id is not null 
      and customer_id is not null 
      and institution_id is not null 
      and loan_amount is not null 
      and loan_status is not null 
      and interest_rate is not null 
      and loan_tenure_months is not null 
      and application_date is not null 
      and disbursement_date is not null
      and maturity_date is not null 
      and emi_amount is not null 
      and purpose_of_loan is not null
      and loan_amount > 0 
      and emi_amount > 0
      and interest_rate > 0
      and application_date < disbursement_date 
      and application_date < maturity_date 
      and disbursement_date < maturity_date
      and loan_amount > 0
      and interest_rate between 5 and 20 
      and loan_tenure_months > 0
      and abs(round(emi_amount) - round(emi_amount_calc)) <= 2

In [0]:
%sql
select * 
from silver.loans

In [0]:
%sql
select loan_id, count(*) 
from silver.loans
group by loan_id 
having count(*) > 1